In [2]:
# Install / upgrade required packages (quiet install)
!pip install -q --upgrade transformers datasets accelerate

# Import libraries and print versions for reproducibility
import transformers, datasets, torch
from transformers import AutoTokenizer, DistilBertForMaskedLM
from datasets import load_dataset, DatasetDict
print("transformers:", transformers.__version__, "datasets:", datasets.__version__, "torch:", torch.__version__)


transformers: 4.57.1 datasets: 4.4.1 torch: 2.8.0+cu126


In [3]:
# Load DistilBERT tokenizer and masked-language-model head
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = DistilBertForMaskedLM.from_pretrained(model_name)

# Quick sanity print
print("Mask token:", tokenizer.mask_token, "Vocab size:", tokenizer.vocab_size)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Mask token: [MASK] Vocab size: 30522


In [4]:
# Load IMDb dataset and use a smaller subset for reliable runs
raw = load_dataset("imdb")
train_small = raw["train"].shuffle(seed=42).select(range(5000))   # use 5000 for demo
test_small = raw["test"].shuffle(seed=42).select(range(1000))     # use 1000 for demo
dataset = DatasetDict({"train": train_small, "test": test_small})
print(dataset)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1000
    })
})


In [5]:
# Tokenize text; do not pad here (we will chunk later)
max_length = 128

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding=False)

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text", "label"])
print(tokenized)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1000
    })
})


In [6]:
# Group tokenized examples into contiguous blocks of max_length
def group_texts(examples):
    concatenated = []
    for ids in examples["input_ids"]:
        concatenated.extend(ids)
    total_length = (len(concatenated) // max_length) * max_length
    result = {
        "input_ids": [concatenated[i : i + max_length] for i in range(0, total_length, max_length)]
    }
    result["attention_mask"] = [[1] * max_length for _ in result["input_ids"]]
    return result

# Process in batches to avoid memory issues
tokenized_blocked = tokenized.map(group_texts, batched=True, batch_size=1000, remove_columns=tokenized["train"].column_names)
print(tokenized_blocked)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 10706
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2046
    })
})


In [7]:
# Data collator will dynamically mask tokens during training (mlm_probability=0.15)
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
print("Data collator prepared.")

Data collator prepared.


In [8]:
# Use TrainingArguments and Trainer; set num_train_epochs=1 and loop manually below for compatibility
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./distilbert_mlm_results",
    overwrite_output_dir=True,
    num_train_epochs=1,                    # will loop externally for multiple epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=3
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_blocked["train"],
    eval_dataset=tokenized_blocked["test"],
    data_collator=data_collator
)

print("Trainer initialized.")

Trainer initialized.


In [ ]:
# Manual epoch loop: trains one epoch per trainer.train() call and evaluates after each epoch
import math, os

NUM_EPOCHS = 3
base_output_dir = training_args.output_dir

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n=== Epoch {epoch}/{NUM_EPOCHS} ===")
    train_result = trainer.train()
    if train_result.metrics:
        print("Training metrics:", train_result.metrics)

    eval_results = trainer.evaluate()
    eval_loss = eval_results.get("eval_loss")
    if eval_loss is not None:
        try:
            perplexity = math.exp(eval_loss) if eval_loss < 100 else float("inf")
        except OverflowError:
            perplexity = float("inf")
        print(f"Eval loss: {eval_loss:.4f} | Perplexity: {perplexity:.2f}")
    else:
        print("Eval results:", eval_results)

    # Save model & tokenizer for this epoch
    epoch_dir = os.path.join(base_output_dir, f"epoch-{epoch}")
    trainer.save_model(epoch_dir)
    tokenizer.save_pretrained(epoch_dir)
    print(f"Saved checkpoint to {epoch_dir}")

print("\nTraining complete.")


=== Epoch 1/3 ===


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ewitcselabbatch2 (ewitcselabbatch2-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,2.807800
